# MarketMinds — Phase 3b: Validate Baseline Models Across the Full Basket

`03_baseline_models.ipynb` found that ARIMA looks skilled in calm markets (66.9% directional accuracy on SENSEX in 2017) but collapses in crises (44.8% in 2008, 31.4% in COVID), while linear regression never showed real skill anywhere. This notebook checks whether that pattern is a SENSEX-specific fluke or holds across the basket, and whether it's stronger for volatile/cyclical stocks than defensive ones.

What's new here vs. the single-asset notebook:
- Same walk-forward linear regression + ARIMA logic, but now imported from `src/forecasting.py` (promoted there since it's now used across notebooks, same pattern as `src/backtest.py`) and looped over every asset in the basket.
- A confidence interval and significance test (vs. 50% random chance) on every directional-accuracy number, since a single-asset test on 51 COVID trading days is thin evidence on its own -- checking whether the pattern holds across ~20 independent assets is what actually makes it credible.
- Sector tags merged in, to compare volatile/cyclical sectors (Auto, Metals, Energy) against defensive ones (FMCG, Pharma). Note: the basket has no airline, so Auto is used as the higher-beta cyclical comparison instead.
- One summary table (`results_df`), saved to `results/`, instead of printing metrics per asset.

**Runtime note**: ~21 assets x 2 models x 3 windows means several hundred ARIMA fits total. Expect this to take a few minutes, not seconds -- progress prints below so you can see it's working.

**Note on execution**: same as always — you run every cell, I don't execute anything. Paste back any errors.

## Setup

In [ ]:
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyarrow.parquet as pq

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)

PROJECT_ROOT = Path('..').resolve()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
RESULTS_DIR = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
sys.path.append(str(PROJECT_ROOT))

from src.forecasting import build_feature_target_table, train_eval_linreg, train_eval_arima, compute_eval_metrics

master = pq.read_table(PROCESSED_DIR / 'marketminds_master.parquet').to_pandas()
symbol_metadata = pq.read_table(PROCESSED_DIR / 'symbol_metadata.parquet').to_pandas()
print('Master shape:', master.shape)

## 1. Define the basket and windows

Same three windows as the single-asset notebook. India VIX is excluded (not investable, not meaningfully forecastable the same way); everything else -- all 19 stocks, both indices, and gold -- is included, since seeing how gold's pattern compares to equities is itself informative given the safe-haven story from Phase 2.

In [ ]:
ASSETS = sorted(t for t in master['ticker'].unique() if t != '^INDIAVIX')

WINDOWS = {
    'normal': {'train_end': '2016-12-31', 'test_start': '2017-01-01', 'test_end': '2017-12-31'},
    '2008_financial_crisis': {'train_end': '2007-12-31', 'test_start': '2008-01-01', 'test_end': '2009-03-31'},
    'covid_crash': {'train_end': '2019-12-31', 'test_start': '2020-02-20', 'test_end': '2020-04-30'},
}

MIN_TRAIN_ROWS = 250

print(len(ASSETS), 'assets:', ASSETS)

## 2. Run walk-forward evaluation for every (asset, window, model) combination

Some assets won't have enough history before a given window's train cutoff -- GOLDBEES.NS before 2008 is the known case, same root cause as the NIFTY 50 issue in the single-asset notebook. Those combinations are skipped and recorded with a reason rather than silently dropped, so the summary table shows exactly what was and wasn't evaluated.

In [ ]:
results = []
start_time = time.time()

for i, ticker in enumerate(ASSETS, 1):
    asset_df = master[master['ticker'] == ticker]
    feature_table = build_feature_target_table(asset_df)
    print(f'[{i}/{len(ASSETS)}] {ticker} -- feature table: {len(feature_table)} rows, elapsed {time.time() - start_time:.0f}s')

    for window_name, cfg in WINDOWS.items():
        train_rows = len(feature_table.loc[:cfg['train_end']])
        test_rows = len(feature_table.loc[cfg['test_start']:cfg['test_end']])

        if train_rows < MIN_TRAIN_ROWS or test_rows == 0:
            for model_name in ['linear_regression', 'arima']:
                results.append({
                    'ticker': ticker, 'window': window_name, 'model': model_name,
                    'status': 'skipped_insufficient_history', 'arima_converged': np.nan,
                    'mae_pct': np.nan, 'rmse_pct': np.nan, 'directional_accuracy_pct': np.nan,
                    'dir_acc_ci_low': np.nan, 'dir_acc_ci_high': np.nan,
                    'p_value_vs_random': np.nan, 'significant_vs_random': False,
                    'significant_better_than_random': False, 'significant_worse_than_random': False,
                    'n_predictions': train_rows,
                })
            continue

        lr_pred, lr_actual = train_eval_linreg(feature_table, cfg)
        lr_metrics = compute_eval_metrics(lr_actual, lr_pred)
        results.append({'ticker': ticker, 'window': window_name, 'model': 'linear_regression', 'status': 'ok', 'arima_converged': np.nan, **lr_metrics})

        ar_pred, ar_actual, ar_converged = train_eval_arima(feature_table, cfg)
        ar_metrics = compute_eval_metrics(ar_actual, ar_pred)
        results.append({'ticker': ticker, 'window': window_name, 'model': 'arima', 'status': 'ok', 'arima_converged': ar_converged, **ar_metrics})
        if not ar_converged:
            print(f'  -- warning: ARIMA did not converge for {ticker} / {window_name}')

print(f'Done in {time.time() - start_time:.0f}s -- {len(results)} rows')
results_df = pd.DataFrame(results)

## 3. Merge in sector metadata

In [ ]:
results_df = results_df.merge(symbol_metadata, left_on='ticker', right_on='ticker', how='left')
cols = ['ticker', 'category', 'window', 'model', 'status', 'arima_converged', 'mae_pct', 'rmse_pct',
        'directional_accuracy_pct', 'dir_acc_ci_low', 'dir_acc_ci_high',
        'p_value_vs_random', 'significant_vs_random',
        'significant_better_than_random', 'significant_worse_than_random', 'n_predictions']
results_df = results_df[cols]
print('Skipped combinations:', (results_df['status'] == 'skipped_insufficient_history').sum())
print('Non-converged ARIMA fits:', (results_df['arima_converged'] == False).sum())
results_df.head(10)

## 4. Save the summary table

In [ ]:
results_path = RESULTS_DIR / 'baseline_model_validation_full_basket.csv'
results_df.to_csv(results_path, index=False)
print('Saved', len(results_df), 'rows to', results_path)

## 5. Does ARIMA's 'skilled in calm markets, collapses in crisis' pattern hold across the basket?

For each asset with valid results in all three windows, check: is `normal` directional accuracy meaningfully above 50%, and does it fall (ideally toward or below 50%) in 2008 and/or COVID? Counting how many assets fit this pattern is what turns the SENSEX finding into a general claim.

In [ ]:
arima_df = results_df[(results_df['model'] == 'arima') & (results_df['status'] == 'ok')]
pivot = arima_df.pivot(index='ticker', columns='window', values='directional_accuracy_pct')
pivot = pivot.dropna(subset=['normal'])

pivot['collapses_2008'] = pivot.get('2008_financial_crisis') < pivot['normal']
pivot['collapses_covid'] = pivot.get('covid_crash') < pivot['normal']
pivot['pattern_holds'] = pivot['collapses_2008'].fillna(False) | pivot['collapses_covid'].fillna(False)

print(f"{pivot['pattern_holds'].sum()} / {len(pivot)} assets show ARIMA directional accuracy falling from normal to at least one crisis window")
pivot.sort_values('normal', ascending=False)

## 6. Sector-level comparison: volatile/cyclical vs defensive

Averaging ARIMA's directional accuracy by sector and window. No airline in this basket, so Auto (MARUTI, TATAMOTORS) stands in as the higher-beta cyclical comparison against the defensive sectors (FMCG, Pharma).

In [ ]:
sector_summary = arima_df.groupby(['category', 'window'])['directional_accuracy_pct'].mean().unstack()
sector_summary = sector_summary[['normal', '2008_financial_crisis', 'covid_crash']]
sector_summary['covid_drop_from_normal'] = sector_summary['normal'] - sector_summary['covid_crash']
sector_summary.sort_values('covid_drop_from_normal', ascending=False)

## 7. Honesty check on statistical significance

The COVID window is only ~51 trading days per asset, so any single asset's directional-accuracy test is individually underpowered (a wide confidence interval, often not `significant_vs_random` on its own). What makes the finding credible isn't any one asset's p-value -- it's whether the *direction* of the effect (lower accuracy in COVID than normal) shows up consistently across many independent assets, which Step 5's count already speaks to. This cell just quantifies how many individual tests actually clear the significance bar, so the write-up doesn't overclaim.

In [ ]:
sig_summary = results_df[results_df['status'] == 'ok'].groupby(['model', 'window']).agg(
    total_assets=('ticker', 'count'),
    significant_better=('significant_better_than_random', 'sum'),
    significant_worse=('significant_worse_than_random', 'sum'),
)
sig_summary['pct_significant_better'] = (sig_summary['significant_better'] / sig_summary['total_assets'] * 100).round(1)
sig_summary['pct_significant_worse'] = (sig_summary['significant_worse'] / sig_summary['total_assets'] * 100).round(1)
sig_summary

## Summary

What this notebook produced:
- `results/baseline_model_validation_full_basket.csv` -- MAE, RMSE, directional accuracy (with 95% CI and p-value vs. random chance), across every asset x window x model combination that had enough history to evaluate
- Step 5's count of how many assets show ARIMA's normal-to-crisis directional-accuracy drop -- the evidence for whether the single-asset finding generalizes
- Step 6's sector breakdown -- whether cyclical sectors collapse harder than defensive ones
- Step 7's significance summary -- how many individual asset-level tests actually clear statistical significance, to keep the write-up honest about the small-sample COVID window

Next: use these results to decide whether the current feature set/models are worth carrying into tree-based models (Random Forest / XGBoost), or whether that's better spent on the assets/windows where the baseline is already known to be weak.